In [1]:
import pandas as pd
import numpy as np
from scipy.stats import chisquare
df = pd.read_csv('/content/marketpele_ab_test.xlsx - data set.csv')
df.head(3)

,date,publisher_id,platform,group_name,pageviews,visible_pageviews,sessions,revenue,sponsord_clicks,organic_clicks
0,2019-03-01,101,Desktop,A,16 580,5 418,12 684,"$ 34,52",300,849
1,2019-03-01,101,Desktop,B,16 191,4 906,12 717,"$ 32,92",268,555
2,2019-03-01,106,Desktop,A,16 227,11 395,11 750,"$ 20,16",601,2425


In [2]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 126 entries, 0 to 125
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype 
---  ------               --------------  ----- 
 0   date                 126 non-null    object
 1   publisher_id         126 non-null    int64 
 2   platform             126 non-null    object
 3   group_name           126 non-null    object
 4    pageviews           126 non-null    object
 5    visible_pageviews   126 non-null    object
 6    sessions            126 non-null    object
 7    revenue             126 non-null    object
 8   sponsord_clicks      126 non-null    int64 
 9   organic_clicks       126 non-null    int64 
dtypes: int64(3), object(7)
memory usage: 10.0+ KB


In [3]:
# Лишнии пробелы в названиях
df.columns = df.columns.str.strip()

# Конвертация числовых колонок
numeric_cols = ['pageviews', 'visible_pageviews', 'sessions', 'revenue']
for col in numeric_cols:
    if df[col].dtype == 'object':
        df[col] = df[col].astype(str).str.replace(r'[^\d.]', '', regex=True).astype(float)

df['RPM'] = df['revenue'] / df['visible_pageviews'] * 1000

df.to_csv('marketpele_ab.csv', index=False)


In [4]:
#Пропуски
df[['revenue', 'visible_pageviews', 'sponsord_clicks', 'organic_clicks']].isnull().sum()

,0
revenue,0
visible_pageviews,0
sponsord_clicks,0
organic_clicks,0


In [5]:
# Дубликаты
key_cols = ['date', 'publisher_id', 'platform', 'group_name']
print(f'Дубликаты по ключу: {df.duplicated(subset=key_cols).sum()}')

Дубликаты по ключу: 0


In [6]:
# Клики не могут превышать просмотры
df['total_clicks'] = df['sponsord_clicks'] + df['organic_clicks']
invalid = df[df['total_clicks'] > df['visible_pageviews']]
print(f"Клики > visible_pageviews: {len(invalid)} строк")

Клики > visible_pageviews: 0 строк


In [7]:
# SRM-тест (баланс количества строк)
n_a = (df['group_name'] == 'A').sum()
n_b = (df['group_name'] == 'B').sum()
chi2, p_srm = chisquare(f_obs=[n_a, n_b], f_exp=[(n_a+n_b)/2, (n_a+n_b)/2])
print(f"SRM-тест: A={n_a}, B={n_b} | p-value={p_srm:.4f}")
# Сплит проведен корректно

SRM-тест: A=63, B=63 | p-value=1.0000


In [8]:
# Баланс по трафику
sess_a = df.loc[df['group_name']=='A', 'sessions'].sum()
sess_b = df.loc[df['group_name']=='B', 'sessions'].sum()
ratio_a = sess_a / (sess_a + sess_b)
print(f"Баланс сессий: A={ratio_a:.1%} | B={1-ratio_a:.1%}")
# небольшое естественное отклонение в поведении пользователей внутри агрегированных строк

Баланс сессий: A=48.0% | B=52.0%
